In [23]:
import os
import numpy as np
from openai import OpenAI
from dotenv import load_dotenv
from fastembed import TextEmbedding
import re
from rank_bm25 import BM25Okapi

load_dotenv()

True

### Chunking

In [24]:
def chunk_by_section(text):
    """Split document by markdown headers (##)"""
    pattern = r"\n## "
    chunks = re.split(pattern, text)
    return [chunk.strip() for chunk in chunks if chunk.strip()]

with open('./report.md','r') as f:
    text = f.read()

useful_chunks = chunk_by_section(text)

### Embeddings

In [25]:
embedding_model = TextEmbedding()


def generate_embedding(texts):
    return list(embedding_model.embed(texts))


chunk_embeddings = [generate_embedding(chunk) for chunk in useful_chunks]

### Vector store

In [26]:
class VectorIndex:
    def __init__(self):
        self.vectors = []
        self.metadata = []

    def add_vector(self, embeddings, metadata):
        self.vectors.append(np.array(embeddings).flatten())
        self.metadata.append(metadata)

    def search(self, query_embedding, top_k=2):
        query_emb = np.array(query_embedding)

        # Calculate similarities
        similarities = []
        for i, vec in enumerate(self.vectors):
            vec = np.array(vec)
            similarity = np.dot(query_emb, vec) / (
                np.linalg.norm(query_emb) * np.linalg.norm(vec)
            )
            similarities.append((similarity, i))

        # Sort by similarity (highest first)
        similarities.sort(reverse=True)

        # Return top_k results
        results = []
        for sim, idx in similarities[:top_k]:
            results.append(
                {
                    "distance": 1 - sim,
                    "content": self.metadata[idx]["content"],
                    "similarity": sim,
                }
            )
        return results


store = VectorIndex()

# Now looping through each chunk and it's embeddings
for embedding, chunk in zip(chunk_embeddings, useful_chunks):
    store.add_vector(embedding, {"content": chunk})

### User quey

In [27]:
user_qn = "What did the software engineering department do last year?"
question_embedding = list(embedding_model.embed([user_qn]))[0]

### BM25 index

In [28]:
tokenized_chunks = [chunk.lower().split() for chunk in useful_chunks]

# BM25 index
bm25 = BM25Okapi(tokenized_chunks)

In [29]:
def hybrid_search(query, chunks, bm25, vector_store, top_k=3):
    # 1. Getting BM25 results
    tokenized_query = query.lower().split()
    bm25_results = bm25.get_top_n(tokenized_query, chunks, n=top_k * 2)

    # 2. Getting semantic results
    query_emb = list(embedding_model.embed([query]))[0]
    semantic_results = vector_store.search(query_emb, top_k=top_k * 2)
    # print(semantic_results)

    # 3. Combining and deduplicating
    combined = {}

    # Adding BM25 results with height
    for i, chunk in enumerate(bm25_results):
        combined[chunk] = combined.get(chunk, 0) + (top_k * 2 - i)

    # Adding semantic results with weight
    for i, result in enumerate(semantic_results):
        chunk = result['content']
        combined[chunk] = combined.get(chunk, 0) + (top_k * 2 - i)

    # Sorting by combines score and returning top_k
    sorted_chunks = sorted(combined.items(), key=lambda x: x[1], reverse=True)
    return [chunk for chunk, score in sorted_chunks[:top_k]]

In [30]:
top_chunks = hybrid_search(
    query="incident-2023 software engineering",
    chunks=useful_chunks,
    bm25=bm25,
    vector_store=store,
)

top_chunks

['Section 2: Software Engineering - Project Phoenix Stability Enhancements\n\nThe Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importance of robust testing protocols, especially given the dependencies identified b

### LLM

In [31]:
context = top_chunks[0]
print(context)

prompt = f"""You are a helpful assistant answering questions based on provided context.

CONTEXT:
{context}

USER QUESTION:
{user_qn}

Answer the question based ONLY on the context provided. If the answer cannot be found in the context, say "I cannot find this information in the provided documents."
"""

Section 2: Software Engineering - Project Phoenix Stability Enhancements

The Software Engineering division dedicated considerable effort to improving the stability and performance of the core systems underpinning Project Phoenix. Recurring issues, particularly `ERR_MEM_ALLOC_FAIL_0x8007000E` during peak loads and `TIMEOUT_QUERY_DB_0xDEADBEEF` affecting data retrieval operations, were prioritized, at a cost of INC-2023-Q4-011. Root cause analysis pointed towards inefficiencies in the primary data caching algorithm and suboptimal database indexing strategies. The deployment of a patch addressed the memory allocation error, resulting in a measured 40% reduction in critical failures under simulated stress tests during Q4 2024 (Test Case ID: INC-2023-Q4-011). Further refactoring of the query module, scheduled for the next release cycle, aims to resolve the timeout issue. These findings underscore the importance of robust testing protocols, especially given the dependencies identified by th

In [32]:
API_KEY=os.getenv("OPENROUTER_API_KEY")
BASE_URL="https://openrouter.ai/api/v1"
MODEL="stepfun/step-3.5-flash:free"

In [33]:
def get_llm_ans(prompt):
    client = OpenAI(
    api_key=API_KEY,
    base_url=BASE_URL
    )
    
    completion = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            },
        ],
    )
    
    return completion.choices[0].message.content



ans = get_llm_ans(prompt)
ans

"Based on the provided context, the Software Engineering department dedicated effort to improving the stability and performance of Project Phoenix's core systems in Q4 2024. Their actions included:\n\n- Prioritizing and addressing recurring errors such as `ERR_MEM_ALLOC_FAIL_0x8007000E` and `TIMEOUT_QUERY_DB_0xDEADBEEF`.\n- Conducting root cause analysis that identified inefficiencies in the data caching algorithm and database indexing.\n- Deploying a patch for the memory allocation error, which resulted in a 40% reduction in critical failures during simulated stress tests.\n- Scheduling refactoring of the query module for a future release to resolve the timeout issue.\n- Assisting with the INC-2023-Q4-011 incident during Q4 2024.\n- Monitoring system telemetry for regressions and emphasizing robust testing protocols, especially given dependencies from other teams.\n\nThese activities were part of Project Phoenix Stability Enhancements, with specific references to testing and incident 